In [41]:
from pathlib import Path

import pandas as pd


def dataset_membership_df(dataset_dir: str | Path) -> pd.DataFrame:
    """Return a dataframe with sample_name plus train/val/test indicator columns."""
    dataset_path = Path(dataset_dir)
    if not any((dataset_path / s).exists() for s in ("train", "val", "test")):
        nested = dataset_path / "dataset"
        if nested.exists():
            dataset_path = nested

    rows: dict[str, dict[str, int]] = {}
    for split in ("train", "val", "test"):
        split_path = dataset_path / split
        if not split_path.exists():
            continue
        for file in split_path.rglob("*"):
            if file.is_file():
                name = file.name
                rows.setdefault(name, {"sample_name": name, "train": 0, "val": 0, "test": 0})
                rows[name][split] = 1

    if not rows:
        return pd.DataFrame(columns=["sample_name", "train", "val", "test"])
    return pd.DataFrame(rows.values()).sort_values("sample_name").reset_index(drop=True)


In [42]:
df = dataset_membership_df("/mnt/slowdisk/public/HIPPA/multiclass_classification")

In [43]:
df.head()

,sample_name,train,val,test
0,RGB_04000_241021_000.jpg,1,0,0
1,RGB_04000_241021_090.jpg,1,0,0
2,RGB_04000_241021_180.jpg,1,0,0
3,RGB_04000_241021_270.jpg,1,0,0
4,RGB_04000_241023_000.jpg,1,0,0


In [44]:
df[df["val"]==1].head()

,sample_name,train,val,test
96,RGB_04004_241021_000.jpg,0,1,0
97,RGB_04004_241021_090.jpg,0,1,0
98,RGB_04004_241021_180.jpg,0,1,0
99,RGB_04004_241021_270.jpg,0,1,0
100,RGB_04004_241023_000.jpg,0,1,0


In [45]:
def dataset_metadata_df(dataset_dir: str | Path) -> pd.DataFrame:
    """Return dataframe with sample metadata plus train/val/test indicators."""
    dataset_path = Path(dataset_dir)
    if not any((dataset_path / s).exists() for s in ("train", "val", "test")):
        nested = dataset_path / "dataset"
        if nested.exists():
            dataset_path = nested

    def _parse(name: str):
        parts = name.split("_")
        modality = parts[0] if parts else None
        segment = parts[1] if len(parts) > 1 else None
        batch = int(segment[:2]) if segment and segment[:2].isdigit() else None
        treatment = int(segment[2]) if segment and len(segment) > 2 and segment[2].isdigit() else None
        apple_id = segment[3:] if segment and len(segment) > 3 else segment
        date = parts[2] if len(parts) > 2 else None
        angle = parts[3].split(".")[0] if len(parts) > 3 else None
        return modality, apple_id, batch, treatment, date, angle

    rows: dict[str, dict[str, object]] = {}
    for split in ("train", "val", "test"):
        split_path = dataset_path / split
        if not split_path.exists():
            continue
        for file in split_path.rglob("*"):
            if file.is_file():
                name = file.name
                modality, apple_id, batch, treatment, date, angle = _parse(name)
                rows.setdefault(
                    name,
                    {
                        "sample_name": name,
                        "apple_id": apple_id,
                        "modality": modality,
                        "batch": batch,
                        "treatment": treatment,
                        "date": date,
                        "angle": angle,
                        "train": 0,
                        "val": 0,
                        "test": 0,
                    },
                )
                rows[name][split] = 1

    if not rows:
        return pd.DataFrame(columns=["sample_name", "apple_id", "modality", "batch", "treatment", "date", "angle", "train", "val", "test"])
    return pd.DataFrame(rows.values()).sort_values("sample_name").reset_index(drop=True)


In [46]:
df_splitted = dataset_metadata_df("/mnt/slowdisk/public/HIPPA/multiclass_classification")

In [47]:
df_splitted

,sample_name,apple_id,modality,batch,treatment,date,angle,train,val,test
0,RGB_04000_241021_000.jpg,00,RGB,4,0,241021,000,1,0,0
1,RGB_04000_241021_090.jpg,00,RGB,4,0,241021,090,1,0,0
2,RGB_04000_241021_180.jpg,00,RGB,4,0,241021,180,1,0,0
3,RGB_04000_241021_270.jpg,00,RGB,4,0,241021,270,1,0,0
4,RGB_04000_241023_000.jpg,00,RGB,4,0,241023,000,1,0,0
...,...,...,...,...,...,...,...,...,...,...
13319,RGB_17823_250702_270.jpg,23,RGB,17,8,250702,270,1,0,0
13320,RGB_17823_250704_000.jpg,23,RGB,17,8,250704,000,1,0,0
13321,RGB_17823_250704_090.jpg,23,RGB,17,8,250704,090,1,0,0
13322,RGB_17823_250704_180.jpg,23,RGB,17,8,250704,180,1,0,0


In [48]:
def check_split_leakage(meta_df: pd.DataFrame | None = None, dataset_dir: str | Path | None = None) -> pd.DataFrame:
    """Detect apples (batch+treatment+apple_id) that land in more than one split.

    Pass either a precomputed dataframe from dataset_metadata_df or a dataset_dir to load one.
    Returns a dataframe of violations with the splits each apple touches.
    """
    if meta_df is None:
        if dataset_dir is None:
            raise ValueError("Provide either meta_df or dataset_dir")
        meta_df = dataset_metadata_df(dataset_dir)
    if meta_df.empty:
        return pd.DataFrame(columns=["batch", "treatment", "apple_id", "splits", "file_count"])

    def splits_present(group: pd.DataFrame) -> list[str]:
        return [s for s in ("train", "val", "test") if group[s].any()]
    violations = []
    for (batch, treatment, apple_id), group in meta_df.groupby(["batch", "treatment", "apple_id"], dropna=False):
        splits = splits_present(group)
        if len(splits) > 1:
            violations.append(
                {
                    "batch": batch,
                    "treatment": treatment,
                    "apple_id": apple_id,
                    "splits": splits,
                    "file_count": len(group),
                }
            )

    if not violations:
        print("No split leakage detected: each apple appears in a single split across train/val/test.")
        return pd.DataFrame(columns=["batch", "treatment", "apple_id", "splits", "file_count"])
    print(f"Found {len(violations)} apples spanning multiple splits.")
    return pd.DataFrame(violations).sort_values(["batch", "treatment", "apple_id"]).reset_index(drop=True)


In [49]:
df_splitted = dataset_metadata_df("/mnt/slowdisk/public/HIPPA/multiclass_classification")  # adjust as needed
leaks = check_split_leakage(meta_df=df_splitted)
leaks


No split leakage detected: each apple appears in a single split across train/val/test.


,batch,treatment,apple_id,splits,file_count


In [51]:
def summarize_splits(meta_df: pd.DataFrame) -> dict:
    """Return split totals and split-by-treatment totals from metadata dataframe."""
    if meta_df.empty:
        return {"split_counts": {}, "treatment_split_counts": pd.DataFrame()}

    split_counts = {split: int(meta_df[split].sum()) for split in ("train", "val", "test")}
    treatment_split_counts = (
        meta_df.groupby("treatment")[['train', 'val', 'test']].sum().astype(int).sort_index()
    )
    return {"split_counts": split_counts, "treatment_split_counts": treatment_split_counts}



summary = summarize_splits(df_splitted)
summary["split_counts"], summary["treatment_split_counts"]


({'train': 10524, 'val': 1324, 'test': 1476},
            train  val  test
 treatment                  
 0           2716  348   432
 1           1248  180   204
 2           1328  148   300
 3            624   72    72
 4            672  144    48
 5           1092  148   200
 6            476   56    84
 7           1212   96    12
 8           1156  132   124)